<a href="https://colab.research.google.com/github/sw030701-ai/motor-control-optimization/blob/main/experiments/01_dc_motor_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · DC Motor Model & Open-Loop Validation
### Literature-Based Nominal Motor Parameters → State-Space Model → Open-Loop Sanity Check

---

### Overview

이 notebook은 PID tuning 전에 먼저 plant인 `DC Motor`를 정의하고 검증한다.

```text
Nominal Motor Parameter Set
      ↓
DC Motor State-Space Model
      ↓
Open-Loop Step Voltage Simulation
      ↓
Theoretical ω_ss와 Simulation 비교
      ↓
Reachable Reference Speed 결정
```

여기서 정한 motor parameter와 reference speed를 다음 notebook인 `02_pid_baseline_tuning.ipynb`에서 사용한다.

In [ ]:
import os, sys, json, platform, subprocess, math
from pathlib import Path


def _in_colab():
    return "google.colab" in sys.modules


def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").exists() and (cand / "docs").exists():
            return cand
    return p


REPO_URL = "https://github.com/sw030701-ai/motor-control-optimization.git"

if _in_colab():
    root = Path("/content/motor-control-optimization")
    if not root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(root)], check=True)
    else:
        subprocess.run(["git", "pull", "--ff-only"], cwd=root, check=False)
    ROOT = root
else:
    ROOT = _find_root(Path.cwd())

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/xdgcache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
if not _in_colab():
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False


def _git(*args):
    try:
        return subprocess.check_output(["git", *args], cwd=ROOT, text=True).strip()
    except Exception:
        return None

ENV = {
    "root": str(ROOT),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "git_commit": _git("rev-parse", "HEAD"),
    "git_dirty": bool(_git("status", "--short")),
}

print(json.dumps(ENV, indent=2, ensure_ascii=False))

In [ ]:
SAVE_ARTIFACTS = True
V_MAX = 12.0
REFERENCE_FRACTION = 0.50
SIMULATION_TIME = 10.0
DT = 0.001

RESULT_TABLE_DIR = Path("results") / "tables"
RESULT_FIGURE_DIR = Path("results") / "figures"
if SAVE_ARTIFACTS:
    RESULT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
    RESULT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("SAVE_ARTIFACTS:", SAVE_ARTIFACTS)
print("V_MAX:", V_MAX)
print("REFERENCE_FRACTION:", REFERENCE_FRACTION)

## Section 1 — Literature-Based Nominal Parameter Set

- **분석 내용**: 논문에서 제시한 DC motor parameter set을 우리 model notation에 맞게 mapping한다.
- **수정 이유**: 이전 parameter set은 repo 안에서 명확한 논문 출처가 없어서, `nominal motor`로 쓰기에는 근거가 약했다.

사용 출처:

- G. Yüksek et al., **"Enhanced DC motor speed regulation using two-degree-of-freedom PID controller tuned by animated oat optimization with simulation and real-time experimental validation"**, *Measurement and Control*, DOI: `10.1177/00202940261442256`.
- 해당 논문의 `Table 1. DC motor parameters`에서 $J$, $b$, $K_e$, $K_t$, $R$, $L$ 값을 가져온다.

우리 프로젝트 notation에서는 논문의 $J$를 $J_m$으로 사용한다.

In [ ]:
from src.motor.dc_motor import NOMINAL_MOTOR_SOURCE, nominal_dc_motor_params

params = nominal_dc_motor_params()

source_table = pd.DataFrame([
    {"Project Symbol": "R", "Paper Symbol": "R", "Meaning": "Armature resistance", "Value": params.R, "Unit": "ohm"},
    {"Project Symbol": "L", "Paper Symbol": "L", "Meaning": "Armature inductance", "Value": params.L, "Unit": "H"},
    {"Project Symbol": "J_m", "Paper Symbol": "J", "Meaning": "Rotor inertia", "Value": params.J_m, "Unit": "kg m^2"},
    {"Project Symbol": "b", "Paper Symbol": "b", "Meaning": "Viscous friction", "Value": params.b, "Unit": "N m s/rad"},
    {"Project Symbol": "K_t", "Paper Symbol": "K_t", "Meaning": "Torque constant", "Value": params.K_t, "Unit": "N m/A"},
    {"Project Symbol": "K_e", "Paper Symbol": "K_e", "Meaning": "Back-EMF constant", "Value": params.K_e, "Unit": "V s/rad"},
])

display(source_table)
print(json.dumps(NOMINAL_MOTOR_SOURCE, indent=2, ensure_ascii=False))

if SAVE_ARTIFACTS:
    source_table.to_csv(RESULT_TABLE_DIR / "nominal_motor_parameters.csv", index=False)
    parameter_record = {
        "source": NOMINAL_MOTOR_SOURCE,
        "parameters": {
            "R": params.R,
            "L": params.L,
            "J_m": params.J_m,
            "b": params.b,
            "K_t": params.K_t,
            "K_e": params.K_e,
        },
    }
    (RESULT_TABLE_DIR / "nominal_motor_parameters.json").write_text(
        json.dumps(parameter_record, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

## Section 2 — DC Motor Equations

현재 plant는 armature-controlled DC motor이다.

Electrical dynamics:

```math
V(t)
=
L\frac{di(t)}{dt}
+
Ri(t)
+
K_e\omega(t)
```

Mechanical dynamics:

```math
J_m\frac{d\omega(t)}{dt}
=
K_ti(t)
-b\omega(t)
-T_L(t)
```

State vector:

```math
x(t)=
\begin{bmatrix}
i(t)\\
\omega(t)
\end{bmatrix}
```

Nominal open-loop validation에서는 $T_L=0$으로 둔다.

## Section 3 — Theoretical No-Load Steady-State Speed

부하가 없는 조건에서 constant voltage $V$를 넣으면 theoretical steady-state speed는 다음과 같다.

```math
\omega_{ss}
=
\frac{V}{K_e+\frac{Rb}{K_t}}
```

이 값과 simulation final speed가 잘 맞는지 확인하면 motor model과 numerical integration이 정상인지 검증할 수 있다.

In [ ]:
omega_ss_max = params.no_load_steady_state_speed(V_MAX)
omega_ref = round(REFERENCE_FRACTION * omega_ss_max, 2)

steady_state_table = pd.DataFrame([{
    "V_max": V_MAX,
    "omega_ss_max_rad_s": omega_ss_max,
    "reference_fraction": REFERENCE_FRACTION,
    "selected_omega_ref_rad_s": omega_ref,
}])

display(steady_state_table)
print(f"Theoretical no-load speed at {V_MAX:.1f} V: {omega_ss_max:.6f} rad/s")
print(f"Selected reference speed: {omega_ref:.2f} rad/s")

## Section 4 — Open-Loop Validation

여기서는 controller 없이 motor에 step voltage를 넣는다.

```text
V(t) = 12 V
      ↓
DC Motor Plant
      ↓
ω(t)
```

PID baseline tuning은 아직 하지 않는다. 이 section은 plant sanity check만 담당한다.

In [ ]:
from src.simulation.pid_simulation import simulate_open_loop

open_loop = simulate_open_loop(
    motor_params=params,
    voltage=V_MAX,
    simulation_time=SIMULATION_TIME,
    dt=DT,
)

open_loop_final = float(open_loop["omega"][-1])
open_loop_error = abs(open_loop_final - omega_ss_max)

open_loop_record = pd.DataFrame([{
    "V_test": V_MAX,
    "simulation_time_s": SIMULATION_TIME,
    "dt_s": DT,
    "omega_ss_theory": omega_ss_max,
    "omega_final_sim": open_loop_final,
    "absolute_error": open_loop_error,
    "relative_error_percent": 100 * open_loop_error / omega_ss_max,
}])

display(open_loop_record)

plt.figure(figsize=(9, 4.5))
plt.plot(open_loop["time"], open_loop["omega"], label="Open-loop speed")
plt.axhline(omega_ss_max, linestyle="--", color="tab:red", label="Theoretical steady-state")
plt.xlabel("Time [s]")
plt.ylabel("Angular speed [rad/s]")
plt.title("Open-Loop Step Voltage Response")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
if SAVE_ARTIFACTS:
    plt.savefig(RESULT_FIGURE_DIR / "open_loop_validation.png", dpi=160)
plt.show()

## Section 5 — Reference Speed Decision

PID tuning에서 사용할 reference speed는 reachable speed 안쪽으로 잡는다.

```math
\omega_{ref}
=
0.50\times\omega_{ss,max}
```

현재 nominal motor와 $V_{max}=12V$ 기준으로 다음 값을 사용한다.

In [ ]:
reference_record = {
    "V_max": V_MAX,
    "omega_ss_max_rad_s": float(omega_ss_max),
    "reference_fraction": REFERENCE_FRACTION,
    "omega_ref_rad_s": float(omega_ref),
    "simulation_time_s": SIMULATION_TIME,
    "dt_s": DT,
}

if SAVE_ARTIFACTS:
    pd.DataFrame([reference_record]).to_csv(RESULT_TABLE_DIR / "reference_selection_record.csv", index=False)
    (RESULT_TABLE_DIR / "reference_selection_record.json").write_text(
        json.dumps(reference_record, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

print(json.dumps(reference_record, indent=2, ensure_ascii=False))

## Final Summary

`01_dc_motor_model.ipynb`에서 확정한 값은 다음과 같다.

```text
Nominal motor source: literature parameter set
V_max                : 12 V
ω_ss,max             : about 25.20 rad/s
ω_ref                : 12.60 rad/s
```

다음 notebook `02_pid_baseline_tuning.ipynb`에서는 이 plant와 reference를 사용해 conventional PID baseline을 tuning한다.